# DBSCAN for Rare Car Theft Pattern Detection

This notebook demonstrates how to use **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** to identify unusual or rare car theft cases that don't belong to any dense cluster.

## Key Concepts
- **DBSCAN**: Density-based clustering algorithm that groups together points in high-density areas
- **Anomaly Detection**: Points labeled as noise (-1) by DBSCAN are considered anomalies
- **Core Points**: Points with at least `min_samples` neighbors within `eps` radius
- **Border Points**: Points within `eps` of a core point but don't have enough neighbors themselves
- **Noise Points**: Points that don't belong to any cluster (anomalies)

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 2. Generate Synthetic Car Theft Dataset

We'll create a realistic synthetic dataset with various features that could be relevant for car theft pattern analysis.

In [ ]:
np.random.seed(42)

n_samples = 1000

# Generate base features for normal theft patterns (clusters)
# Cluster 1: Night thefts in residential areas (40% of data)
n_cluster1 = 400
cluster1_time = np.random.normal(2, 1, n_cluster1) % 24  # Night hours (10pm-6am)
cluster1_time = np.clip(cluster1_time, 20, 23) if cluster1_time > 12 else np.clip(cluster1_time, 0, 6)
cluster1_value = np.random.normal(15000, 3000, n_cluster1)  # Lower value cars
cluster1_age = np.random.normal(10, 2, n_cluster1)  # Older cars
cluster1_security = np.random.randint(1, 4, n_cluster1)  # Low security
cluster1_location_x = np.random.normal(2, 0.5, n_cluster1)
cluster1_location_y = np.random.normal(2, 0.5, n_cluster1)

# Cluster 2: Day thefts in commercial areas (35% of data)
n_cluster2 = 350
cluster2_time = np.random.normal(14, 2, n_cluster2)  # Afternoon hours
cluster2_value = np.random.normal(35000, 8000, n_cluster2)  # Mid-range cars
cluster2_age = np.random.normal(5, 1.5, n_cluster2)  # Newer cars
cluster2_security = np.random.randint(3, 6, n_cluster2)  # Medium security
cluster2_location_x = np.random.normal(8, 1, n_cluster2)
cluster2_location_y = np.random.normal(7, 1, n_cluster2)

# Cluster 3: Evening thefts near entertainment districts (15% of data)
n_cluster3 = 150
cluster3_time = np.random.normal(21, 1.5, n_cluster3)  # Evening hours
cluster3_value = np.random.normal(50000, 10000, n_cluster3)  # Luxury cars
cluster3_age = np.random.normal(2, 1, n_cluster3)  # Very new cars
cluster3_security = np.random.randint(5, 8, n_cluster3)  # High security
cluster3_location_x = np.random.normal(5, 0.8, n_cluster3)
cluster3_location_y = np.random.normal(9, 0.8, n_cluster3)

# Anomalies: Rare/unusual theft patterns (10% of data)
n_anomalies = 100
# Various unusual patterns
anomaly_time = np.random.uniform(0, 24, n_anomalies)
anomaly_value = np.random.uniform(5000, 150000, n_anomalies)
anomaly_age = np.random.uniform(1, 20, n_anomalies)
anomaly_security = np.random.randint(1, 10, n_anomalies)
anomaly_location_x = np.random.uniform(0, 10, n_anomalies)
anomaly_location_y = np.random.uniform(0, 10, n_anomalies)

# Combine all data
data = {
    'hour_of_day': np.concatenate([cluster1_time, cluster2_time, cluster3_time, anomaly_time]),
    'car_value_usd': np.concatenate([cluster1_value, cluster2_value, cluster3_value, anomaly_value]),
    'car_age_years': np.concatenate([cluster1_age, cluster2_age, cluster3_age, anomaly_age]),
    'security_level': np.concatenate([cluster1_security, cluster2_security, cluster3_security, anomaly_security]),
    'location_x': np.concatenate([cluster1_location_x, cluster2_location_x, cluster3_location_x, anomaly_location_x]),
    'location_y': np.concatenate([cluster1_location_y, cluster2_location_y, cluster3_location_y, anomaly_location_y])
}

df = pd.DataFrame(data)

# Add derived features
df['time_category'] = pd.cut(df['hour_of_day'], 
                              bins=[0, 6, 12, 18, 24], 
                              labels=['Night', 'Morning', 'Afternoon', 'Evening'])
df['is_anomaly_actual'] = [0] * (n_cluster1 + n_cluster2 + n_cluster3) + [1] * n_anomalies

print(f"Dataset Shape: {df.shape}")
print(f"\nFeature Statistics:")
print(df.describe())
print(f"\nActual Anomalies: {df['is_anomaly_actual'].sum()} ({df['is_anomaly_actual'].mean()*100:.1f}%)")

## 3. Data Visualization - Initial Exploration

In [ ]:
# Pairplot of key features
plt.figure(figsize=(14, 10))
pairplot_data = df[['hour_of_day', 'car_value_usd', 'car_age_years', 'security_level']].copy()
pairplot_data['Actual Class'] = pairplot_data.apply(
    lambda x: 'Anomaly' if df.loc[x.name, 'is_anomaly_actual'] == 1 else 'Normal', 
    axis=1
)
sns.pairplot(pairplot_data, hue='Actual Class', plot_kws={'alpha': 0.6, 's': 30})
plt.suptitle('Feature Relationships - Normal vs Anomalous Thefts', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of each feature
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

features = ['hour_of_day', 'car_value_usd', 'car_age_years', 'security_level', 'location_x', 'location_y']
titles = ['Hour of Day', 'Car Value (USD)', 'Car Age (Years)', 'Security Level', 'Location X', 'Location Y']

for idx, (feat, title) in enumerate(zip(features, titles)):
    ax = axes[idx // 3, idx % 3]
    sns.histplot(data=df, x=feat, hue='is_anomaly_actual', ax=ax, alpha=0.6, bins=30)
    ax.set_title(f'Distribution of {title}', fontsize=12)
    ax.set_xlabel(title)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Select features for clustering
feature_columns = ['hour_of_day', 'car_value_usd', 'car_age_years', 'security_level', 'location_x', 'location_y']
X = df[feature_columns].values

# Scale the data (important for DBSCAN)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features scaled. Shape: {X_scaled.shape}")
print(f"Mean after scaling: {X_scaled.mean(axis=0).round(4)}")
print(f"Std after scaling: {X_scaled.std(axis=0).round(4)}")

## 5. Finding Optimal DBSCAN Parameters

DBSCAN requires two parameters:
- **eps (ε)**: Maximum distance between points to be considered neighbors
- **min_samples**: Minimum number of points to form a dense region

We'll use the k-distance graph to find optimal eps.

In [ ]:
# K-distance graph to find optimal eps
def find_optimal_eps(X_scaled, k=4):
    """Plot k-distance graph to help determine optimal eps value"""
    neighbors = NearestNeighbors(n_neighbors=k)
    neighbors_fit = neighbors.fit(X_scaled)
    distances, indices = neighbors_fit.kneighbors(X_scaled)
    
    # Sort distances in descending order
    k_distances = distances[:, k-1]
    k_distances_sorted = np.sort(k_distances)[::-1]
    
    return k_distances_sorted

# Plot k-distance graph
plt.figure(figsize=(12, 6))
k_dist = find_optimal_eps(X_scaled, k=4)
plt.plot(range(len(k_dist)), k_dist, 'b-', linewidth=1)
plt.axhline(y=1.0, color='r', linestyle='--', label='Suggested eps=1.0')
plt.axhline(y=1.5, color='g', linestyle='--', label='Suggested eps=1.5')
plt.xlabel('Points (sorted by distance)')
plt.ylabel(f'{4}-th Nearest Neighbor Distance')
plt.title('K-Distance Graph for Optimal Eps Selection')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Look for the "elbow" point
print("The elbow in the k-distance graph indicates a good eps value.")
print("Points above this threshold are likely outliers.")

## 6. Apply DBSCAN Clustering

In [ ]:
# Apply DBSCAN with different eps values to compare
eps_values = [0.8, 1.0, 1.2, 1.5]
min_samples = 10

results = {}

for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=min_samples, metric='euclidean')
    labels = db.fit_predict(X_scaled)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    results[eps] = {
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'noise_percentage': (n_noise / len(labels)) * 100,
        'labels': labels
    }
    
    print(f"eps={eps:.1f}: Clusters={n_clusters}, Noise points={n_noise} ({results[eps]['noise_percentage']:.1f}%)")

In [ ]:
# Choose the best eps (1.2 seems reasonable for this dataset)
best_eps = 1.2
db_best = DBSCAN(eps=best_eps, min_samples=min_samples, metric='euclidean')
df['cluster'] = db_best.fit_predict(X_scaled)

# Analyze clustering results
n_clusters = df['cluster'].nunique() - (1 if -1 in df['cluster'].values else 0)
n_anomalies_detected = (df['cluster'] == -1).sum()

print(f"\n=== DBSCAN Results (eps={best_eps}, min_samples={min_samples}) ===")
print(f"Total clusters found: {n_clusters}")
print(f"Anomalies detected (noise): {n_anomalies_detected}")
print(f"Anomaly rate: {(n_anomalies_detected/len(df))*100:.2f}%")
print(f"\nCluster distribution:")
print(df['cluster'].value_counts().sort_index())

## 7. Visualization of DBSCAN Results

In [ ]:
# 2D scatter plots showing clusters
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Create color map for clusters (noise points in red)
cluster_colors = df['cluster'].apply(
    lambda x: 'red' if x == -1 else plt.cm.tab10(x % 10)
)

# Plot 1: Location-based clustering
ax1 = axes[0, 0]
scatter1 = ax1.scatter(df['location_x'], df['location_y'], 
                       c=df['cluster'], cmap='tab10', 
                       alpha=0.6, s=50, edgecolors='white', linewidth=0.5)
ax1.set_xlabel('Location X')
ax1.set_ylabel('Location Y')
ax1.set_title('Spatial Clustering (Location)')
ax1.grid(True, alpha=0.3)

# Plot 2: Time vs Value
ax2 = axes[0, 1]
scatter2 = ax2.scatter(df['hour_of_day'], df['car_value_usd'], 
                       c=df['cluster'], cmap='tab10', 
                       alpha=0.6, s=50, edgecolors='white', linewidth=0.5)
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Car Value (USD)')
ax2.set_title('Time vs Value Clustering')
ax2.grid(True, alpha=0.3)

# Plot 3: Car Age vs Security Level
ax3 = axes[1, 0]
scatter3 = ax3.scatter(df['car_age_years'], df['security_level'], 
                       c=df['cluster'], cmap='tab10', 
                       alpha=0.6, s=50, edgecolors='white', linewidth=0.5)
ax3.set_xlabel('Car Age (Years)')
ax3.set_ylabel('Security Level')
ax3.set_title('Car Characteristics Clustering')
ax3.grid(True, alpha=0.3)

# Plot 4: PCA-reduced visualization
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

ax4 = axes[1, 1]
scatter4 = ax4.scatter(X_pca[:, 0], X_pca[:, 1], 
                       c=df['cluster'], cmap='tab10', 
                       alpha=0.6, s=50, edgecolors='white', linewidth=0.5)
ax4.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
ax4.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
ax4.set_title('PCA-Reduced Feature Space')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Highlight anomalies specifically
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

anomalies = df[df['cluster'] == -1]
normal = df[df['cluster'] != -1]

# Plot 1: Location
ax1 = axes[0]
ax1.scatter(normal['location_x'], normal['location_y'], 
            alpha=0.3, s=30, c='gray', label='Normal')
ax1.scatter(anomalies['location_x'], anomalies['location_y'], 
            alpha=0.8, s=60, c='red', label='Anomaly', marker='x')
ax1.set_xlabel('Location X')
ax1.set_ylabel('Location Y')
ax1.set_title('Anomalies in Spatial Context')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Time vs Value
ax2 = axes[1]
ax2.scatter(normal['hour_of_day'], normal['car_value_usd'], 
            alpha=0.3, s=30, c='gray', label='Normal')
ax2.scatter(anomalies['hour_of_day'], anomalies['car_value_usd'], 
            alpha=0.8, s=60, c='red', label='Anomaly', marker='x')
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Car Value (USD)')
ax2.set_title('Anomalies in Time-Value Space')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Age vs Security
ax3 = axes[2]
ax3.scatter(normal['car_age_years'], normal['security_level'], 
            alpha=0.3, s=30, c='gray', label='Normal')
ax3.scatter(anomalies['car_age_years'], anomalies['security_level'], 
            alpha=0.8, s=60, c='red', label='Anomaly', marker='x')
ax3.set_xlabel('Car Age (Years)')
ax3.set_ylabel('Security Level')
ax3.set_title('Anomalies in Car Characteristics')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Analyze Detected Anomalies

In [ ]:
# Evaluate detection performance
true_positives = ((df['cluster'] == -1) & (df['is_anomaly_actual'] == 1)).sum()
false_positives = ((df['cluster'] == -1) & (df['is_anomaly_actual'] == 0)).sum()
true_negatives = ((df['cluster'] != -1) & (df['is_anomaly_actual'] == 0)).sum()
false_negatives = ((df['cluster'] != -1) & (df['is_anomaly_actual'] == 1)).sum()

precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("=== Anomaly Detection Performance ===")
print(f"True Positives (correctly identified anomalies): {true_positives}")
print(f"False Positives (normal flagged as anomaly): {false_positives}")
print(f"True Negatives (correctly identified normal): {true_negatives}")
print(f"False Negatives (anomalies missed): {false_negatives}")
print(f"\nPrecision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1_score:.4f}")

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

plt.figure(figsize=(8, 6))
cm = confusion_matrix(df['is_anomaly_actual'], df['cluster'].apply(lambda x: 1 if x == -1 else 0))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Anomaly'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix - Anomaly Detection')
plt.tight_layout()
plt.show()

In [ ]:
# Detailed analysis of detected anomalies
print("=== Characteristics of Detected Anomalies ===")
print(f"\nTotal anomalies detected: {len(anomalies)}")
print(f"\nAnomaly Statistics:")
print(anomalies[feature_columns].describe())

# Show sample anomalies
print("\n=== Sample Anomalous Cases (first 10) ===")
sample_anomalies = anomalies[['hour_of_day', 'car_value_usd', 'car_age_years', 'security_level', 'location_x', 'location_y']].head(10)
print(sample_anomalies.to_string(index=True))

In [ ]:
# Compare anomalies vs normal patterns
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

features_compare = ['hour_of_day', 'car_value_usd', 'car_age_years', 'security_level']
titles = ['Hour of Theft', 'Car Value (USD)', 'Car Age (Years)', 'Security System Level']

for idx, (feat, title) in enumerate(zip(features_compare, titles)):
    ax = axes[idx // 2, idx % 2]
    
    # Box plots
    data_to_plot = [normal[feat], anomalies[feat]]
    bp = ax.boxplot(data_to_plot, labels=['Normal', 'Anomaly'], patch_artist=True)
    
    # Color the boxes
    colors = ['#4CAF50', '#F44336']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title(f'{title} - Normal vs Anomalous', fontsize=12)
    ax.set_ylabel(feat.replace('_', ' ').title())
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Cluster Analysis

In [ ]:
# Analyze each cluster's characteristics
print("=== Cluster Profiles ===")

for cluster_id in sorted(df['cluster'].unique()):
    if cluster_id == -1:
        continue
    
    cluster_data = df[df['cluster'] == cluster_id]
    print(f"\n{'='*50}")
    print(f"CLUSTER {cluster_id}")
    print(f"{'='*50}")
    print(f"Size: {len(cluster_data)} thefts ({len(cluster_data)/len(df)*100:.1f}%)")
    print(f"\nFeature Means:")
    for feat in feature_columns:
        print(f"  {feat}: {cluster_data[feat].mean():.2f} (std: {cluster_data[feat].std():.2f})")
    
    # Characterize the cluster
    avg_hour = cluster_data['hour_of_day'].mean()
    avg_value = cluster_data['car_value_usd'].mean()
    avg_age = cluster_data['car_age_years'].mean()
    
    if avg_hour < 6 or avg_hour > 20:
        time_pattern = "Night thefts"
    elif 10 <= avg_hour <= 16:
        time_pattern = "Daytime thefts"
    else:
        time_pattern = "Evening thefts"
    
    if avg_value < 20000:
        value_pattern = "Low-value vehicles"
    elif avg_value < 50000:
        value_pattern = "Mid-range vehicles"
    else:
        value_pattern = "High-value/Luxury vehicles"
    
    if avg_age > 8:
        age_pattern = "Older vehicles"
    elif avg_age > 4:
        age_pattern = "Mid-age vehicles"
    else:
        age_pattern = "Newer vehicles"
    
    print(f"\nCluster Profile: {time_pattern} targeting {value_pattern} ({age_pattern})")

In [ ]:
# Visualize cluster centers
cluster_centers = df[df['cluster'] != -1].groupby('cluster')[feature_columns].mean()

plt.figure(figsize=(12, 8))
sns.heatmap(cluster_centers.T, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5)
plt.title('Cluster Centers - Feature Means')
plt.ylabel('Features')
plt.xlabel('Cluster ID')
plt.tight_layout()
plt.show()

## 10. Parameter Sensitivity Analysis

In [ ]:
# Test different parameter combinations
eps_range = [0.8, 1.0, 1.2, 1.5, 1.8]
min_samples_range = [5, 10, 15, 20]

sensitivity_results = []

for eps in eps_range:
    for min_s in min_samples_range:
        db = DBSCAN(eps=eps, min_samples=min_s)
        labels = db.fit_predict(X_scaled)
        
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        # Calculate detection metrics
        detected_anomalies = (labels == -1)
        tp = ((detected_anomalies) & (df['is_anomaly_actual'] == 1)).sum()
        fp = ((detected_anomalies) & (df['is_anomaly_actual'] == 0)).sum()
        fn = ((~detected_anomalies) & (df['is_anomaly_actual'] == 1)).sum()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        sensitivity_results.append({
            'eps': eps,
            'min_samples': min_s,
            'n_clusters': n_clusters,
            'n_anomalies': n_noise,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        })

sensitivity_df = pd.DataFrame(sensitivity_results)

# Display as heatmap
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
pivot_f1 = sensitivity_df.pivot('min_samples', 'eps', 'f1_score')
sns.heatmap(pivot_f1, annot=True, fmt='.3f', cmap='YlGnBu')
plt.title('F1 Score by Parameters')
plt.xlabel('eps')
plt.ylabel('min_samples')

plt.subplot(1, 2, 2)
pivot_recall = sensitivity_df.pivot('min_samples', 'eps', 'recall')
sns.heatmap(pivot_recall, annot=True, fmt='.3f', cmap='YlGnBu')
plt.title('Recall by Parameters')
plt.xlabel('eps')
plt.ylabel('min_samples')

plt.tight_layout()
plt.show()

# Best parameters
best_row = sensitivity_df.loc[sensitivity_df['f1_score'].idxmax()]
print(f"\n=== Best Parameters ===")
print(f"eps: {best_row['eps']}")
print(f"min_samples: {best_row['min_samples']}")
print(f"F1 Score: {best_row['f1_score']:.4f}")
print(f"Precision: {best_row['precision']:.4f}")
print(f"Recall: {best_row['recall']:.4f}")

## 11. Real-World Interpretation

In [ ]:
# Generate actionable insights
print("="*60)
print("ACTIONABLE INSIGHTS FOR LAW ENFORCEMENT")
print("="*60)

print("\n### Common Theft Patterns (Clusters) ###")
for cluster_id in sorted(df['cluster'].unique()):
    if cluster_id == -1:
        continue
    
    cluster_data = df[df['cluster'] == cluster_id]
    print(f"\nPattern {cluster_id}: {len(cluster_data)} cases")
    print(f"  - Typical time: {cluster_data['hour_of_day'].mean():.1f}:00 hours")
    print(f"  - Typical target: ${cluster_data['car_value_usd'].mean():,.0f} vehicles")
    print(f"  - Vehicle age: {cluster_data['car_age_years'].mean():.1f} years")
    print(f"  - Location centroid: ({cluster_data['location_x'].mean():.2f}, {cluster_data['location_y'].mean():.2f})")

print("\n### Rare/Unusual Patterns (Anomalies) ###")
print(f"Total unusual cases: {len(anomalies)}")
print("\nThese cases don't fit typical patterns and may indicate:")
print("  - Organized crime with specific high-value targets")
print("  - Insider jobs (dealership thefts)")
print("  - New emerging theft trends")
print("  - Sophisticated theft rings with unusual MO")

print("\n### Recommended Actions ###")
print("1. Increase patrols in identified cluster hotspots during peak hours")
print("2. Investigate anomalous cases for potential organized crime links")
print("3. Alert owners of high-value, newer vehicles about increased risk")
print("4. Review security requirements for vehicles in high-risk categories")

## 12. Save Results

In [ ]:
# Save the dataset with predictions
output_df = df.copy()
output_df['is_anomaly_detected'] = (output_df['cluster'] == -1).astype(int)
output_df.to_csv('car_theft_dbscan_results.csv', index=False)
print(f"Results saved to 'car_theft_dbscan_results.csv'")
print(f"\nFinal dataset shape: {output_df.shape}")
print(f"Columns: {list(output_df.columns)}")

In [ ]:
# Display final summary
print("\n" + "="*60)
print("DBSCAN ANOMALY DETECTION - SUMMARY")
print("="*60)
print(f"Total cases analyzed: {len(df)}")
print(f"Clusters identified: {n_clusters}")
print(f"Anomalies detected: {n_anomalies_detected} ({n_anomalies_detected/len(df)*100:.2f}%)")
print(f"\nDetection Performance:")
print(f"  Precision: {precision:.2%}")
print(f"  Recall: {recall:.2%}")
print(f"  F1 Score: {f1_score:.2%}")
print("\nDBSCAN successfully identified rare theft patterns that don't conform to typical clusters.")